# 01 — Your First Agent

## Objectives

- Build an `Agent` with `instructions` and run it with `await agent.run(...)`
- Read the result: `result.output`, `result.usage`, `result.all_messages()`
- Recognize the **v2 traps** that make every pre-2026 tutorial wrong
- Watch your first trace in Logfire

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## The idea

An `Agent` is three decisions bundled together:

1. **Which model** — a prefixed string like `'openai:gpt-6.1-sol'` (v2 requires the prefix;
   bare names raise `UserError`)
2. **What it should do** — `instructions=` (v2 docs prefer this over `system_prompt=`,
   especially once you carry message history or run multiple agents)
3. *(later notebooks)* **What it can call** — tools, dependencies, output types

Let's build the simplest possible one and actually run it.

In [ ]:
from pydantic_ai import Agent

tutor = Agent(
    MODEL,
    instructions="You are a patient programming tutor. Answer with exactly one clear sentence.",
)

result = await tutor.run("What is an AI agent, really?")
print(result.output)

> ✅ **Checkpoint** — you got a one-sentence answer, and you *paid* for it (fractions of a cent).
> You may also have noticed timestamped lines like `20:29:33.538 agent run` in the console —
> that's Logfire's live console output.
>
> 🔭 If you ran `logfire auth`: open the **Live view** at
> [logfire.pydantic.dev](https://logfire.pydantic.dev), find this run, and expand it. Every model
> call shows a coin badge with tokens and USD cost.

## Inspect the result — and learn the v2 traps

Most blog posts about pydantic-ai are wrong now. The framework went **1.0 in September 2025**
and **2.0 in June 2026**; here's what changed that you'll actually touch:

| Old tutorial says | v2 reality |
|---|---|
| `result.data` | `result.output` |
| `result.usage()` | `result.usage` — a **property** |
| `Agent('gpt-4o')` | needs a prefix: `Agent('openai:gpt-6.1-sol')` |
| `system_prompt=` everywhere | prefer `instructions=` |
| `Agent(instrument=True)` | removed — `logfire.instrument_pydantic_ai()` instead |

In [ ]:
print("output       :", result.output)
print("messages     :", len(result.all_messages()), "messages in this run")
print("usage        :", result.usage)
print("input tokens :", result.usage.input_tokens)
print("output tokens:", result.usage.output_tokens)
print("cost (USD)   :", result.usage.cost)  # may be None if your model's price is unknown

> ✅ **Checkpoint** — `cost` is a real number in dollars. This is your meter running:
> every notebook from here on prints it, so you always know what a run costs.

## What just happened, message-wise

An agent run is just a list of **messages** — the model has no hidden state.
`result.all_messages()` gives you the entire exchange, which is exactly what you'll feed
back in notebook 07 to have "memory".

In [ ]:
for message in result.all_messages():
    print(type(message).__name__, "→", [type(part).__name__ for part in message.parts])

> ✅ **Checkpoint** — you should see a `ModelRequest` (your prompt) and a `ModelResponse`
> (the model's answer), each holding typed *parts*. Messages-in, messages-out: that's the
> whole mental model. Everything else (history, hand-offs, servers) is plumbing around it.

## 🏋️ Exercise — the pirate agent

Build `pirate_agent` so that it answers **any** question in full pirate dialect
("Arrr, the answer be..."). Then:

1. Run it on a question of your choice and print the output.
2. Print `result.usage.output_tokens` — how many tokens did the dialect cost you?
3. Stretch: run the same question twice. Do you get identical answers? Compare the token counts.

Try it in a scratch cell (or a new notebook) before peeking at
[`../solutions/01_pirate_agent.ipynb`](../solutions/01_pirate_agent.ipynb).

## Next

→ **[02_models_and_providers.ipynb](02_models_and_providers.ipynb)** — model strings, settings,
fallbacks, and comparing what different models charge you.